In [0]:
%run /Configurations/Init_Scripts

In [0]:
import os
from datetime import datetime

In [0]:
dbutils.widgets.text("InvoiceDate", "")
InvoiceDate = dbutils.widgets.get("InvoiceDate")

dbutils.widgets.text("DateInterval", "1")
DateInterval = dbutils.widgets.get("DateInterval")

assertion_errors = []

TestCaseDetails = {"validateInvoiceAddendumDetailSnapshotView": {
                    "ErrorMessage": "InvoiceAddendum_v and invoiceaddendumdetailssnapshot_v was not matching",
                    "TestCaseDescription": "Validate cycle count, visit count, invoice amount across all ShipTo’s and Visit classifications between invoice details and summary views"}}

In [0]:
def add_Prefix_ColumnName(df,prefix):
    new_cols = [col(col_name).alias(prefix+col_name) for col_name in df.columns]
    df_Source = df.select(new_cols)
    return df_Source

In [0]:
# adding the function for logging in promotion log table.
dst_TestExecutionResult = "/mnt/silver/TestExecutionResult"


def logIntoTestExecutionResultTable(log_dict):

    df_Source = (
        spark.createDataFrame(log_dict)
        .withColumn("CreatedBy", lit("ADB_AutomationTestCase"))
        .withColumn("UpdatedBy", lit("ADB_AutomationTestCase"))
        .withColumn("Createddate", current_timestamp())
        .withColumn("Updateddate", current_timestamp())
    )

    df_Source.write.format("delta").mode("append").save(dst_TestExecutionResult)

In [0]:
def runAssert(condition, message, TestCaseName, InvoiceDate,MismatchedData):
    try:
        assert condition, message
        print(f"\n \t TestCaseName: {TestCaseName}; TestCaseStatus: Passed")
        log_dict = [{
                    "TestCaseName" : TestCaseName,
                    "TestCaseDescription" : TestCaseDetails[TestCaseName]["TestCaseDescription"],
                    "InvoiceMonth" : str(InvoiceDate),
                    "ValidationType" : "Post Validation Checks",
                    "Status" : "Passed"
                    }]
        logIntoTestExecutionResultTable(log_dict)
    except AssertionError as e:
        assertion_errors.append(str(e))
        print(f"\n \t TestCaseName:{TestCaseName}; TestCaseStatus: Failed")
        log_dict = [{
                    "TestCaseName" : TestCaseName,
                    "TestCaseDescription" : TestCaseDetails[TestCaseName]["TestCaseDescription"],
                    "TestCaseErrorMessage": message,
                    "InvoiceMonth" : str(InvoiceDate),
                    "ValidationType" : "Post Validation Checks",
                    "MismatchedData": str(MismatchedData),
                    "Status" : "Failed"
                    }]
        logIntoTestExecutionResultTable(log_dict)

# Post Validation Test Cases

## 1.Validate cycle count, visit count and invoice amount across all ShipTo’s and Visit classifications between invoice details and summary views.

In [0]:
def validateInvoiceAddendumDetailSnapshotView(InvoiceDate,TestCaseName):
    #Columns that was used to compare
    hashcols = ['TotalCycleCount','TotalVisitCount','InvoiceAmount']

    df_InvoiceDetailsSnapshot = spark.sql(f"""select ShipToAccountId,
                                            PatientClassificationName,
                                            Sum(CycleCount) as TotalCycleCount,
                                            Sum(TotalVisitCount) as TotalVisitCount,
                                            sum(IncrementalInvoiceCharge) as InvoiceAmount,
                                            PromotionName,
                                            SoldToAccountID
                                    from goldzone.invoiceaddendumdetailssnapshot_v where InvoiceCalculationDate = '{InvoiceDate}'
                                    Group By ShipToAccountId,PatientClassificationName,PromotionName,SoldToAccountID""")\
                                    .withColumn("HashKey",sha2(concat_ws('', *hashcols),256))

    df_InvoiceAddendum = spark.sql(f"""select ShipToAccountId,
                                            PatientClassificationName,
                                            abs(TotalCycleCount) as TotalCycleCount,
                                            InvoiceAmount,
                                            abs(TotalVisitCount) as TotalVisitCount,
                                            PromotionName,
                                            SoldToAccountID
                                    from goldzone.invoiceaddendum_v 
                                    where InvoiceCalculationDate = '{InvoiceDate}'""")\
                                    .withColumn("HashKey",sha2(concat_ws('', *hashcols),256))   

    #Added Prefix in Dataframe Columns
    df_InvoiceDetailsSnapshot = add_Prefix_ColumnName(df_InvoiceDetailsSnapshot,"SnapshotView_")
    df_InvoiceAddendum = add_Prefix_ColumnName(df_InvoiceAddendum,"AddendumView_")

    #Validate InvoiceAddendumdetailsnapshot and InvoiceAddendum Summary View
    df_InvoiceDetailsSnapshotResult = df_InvoiceDetailsSnapshot.alias("Snapshot").join(df_InvoiceAddendum.alias("Addendum"),(col("Snapshot.SnapshotView_ShipToAccountId") == col("Addendum.AddendumView_ShipToAccountId")) & (col("Snapshot.SnapshotView_PatientClassificationName") == col("Addendum.AddendumView_PatientClassificationName")) & (col("Snapshot.SnapshotView_PromotionName") == col("Addendum.AddendumView_PromotionName")) & (col("Snapshot.SnapshotView_SoldToAccountID") == col("Addendum.AddendumView_SoldToAccountID")),"FULL")\
                                .withColumn("Comments",when(col("Snapshot.SnapshotView_HashKey") == col("Addendum.AddendumView_HashKey"),"Matched")
                                                        .when(col("Snapshot.SnapshotView_HashKey") != col("Addendum.AddendumView_HashKey"),"Not Matched")
                                                        .when(col("Snapshot.SnapshotView_HashKey").isNull(),"Record was Missing in InvoiceAddendumDetailsSnapshot_v")
                                                        .when(col("Addendum.AddendumView_HashKey").isNull(),"Record was Missing in InvoiceAddendum_v")
                                                        .otherwise("Unknown"))

    ExceptionCount = df_InvoiceDetailsSnapshotResult.filter(col("Comments")!= 'Matched').count()
    MismatchedData = df_InvoiceDetailsSnapshotResult.filter("Comments != 'Matched'").toJSON().collect()
    
    runAssert(ExceptionCount == 0,TestCaseDetails[TestCaseName]["ErrorMessage"],TestCaseName,InvoiceDate,MismatchedData)
    print(f"Exception Count:{ExceptionCount}")
    print("Comparison between goldzone.invoiceaddendum_v and goldzone.invoiceaddendumdetailssnapshot_v:")
    df_InvoiceDetailsSnapshotResult.filter(col("Comments")!= 'Matched').display()

## Execute Test Cases

In [0]:
if InvoiceDate == '':
    InvoiceDate  = (spark.read.table('promotion.dim_invoicecyclemonth')
                          .filter(col('InvoiceDate').between(date_sub(current_date(), lit(DateInterval)),date_add(current_date(), lit(DateInterval))))
                          .select('InvoiceDate').orderBy(desc('InvoiceDate'))
                           .first()[0]
                          )

print(f"InvoiceDate:{InvoiceDate}")

validateInvoiceAddendumDetailSnapshotView(InvoiceDate,"validateInvoiceAddendumDetailSnapshotView")

In [0]:
# Check if there were any Failures in Test Cases
if assertion_errors:
    raise Exception("Failed Test Cases: " + "; ".join(set(assertion_errors)))
print("All Test Cases are passed.")